# Registry, Portable Files, and CLI Workflows

**Model.** This notebook uses the registry and portable model layer rather than focusing on one Hamiltonian. It summarizes available builders, creates an inspectable model file, exercises command-line spectra and plotting, and exports a deterministic artifact bundle.

**Typical uses.** Programmatic model discovery, portable model creation, command-line smoke tests, reproducible figures, and model or matrix interchange without writing a separate script.

**Parameters.** Registry rows expose model categories, basis conventions, dimensions, return types, defaults, sparse support, and validation status. The CLI accepts model parameters and file-oriented commands such as `create`, `inspect`, `spectrum`, `plot`, and `export`.

**Useful plots.** Use `plot_interaction_graph` for portable physical-system data and CLI `plot` for quick spectrum PNG output.


In [1]:
import subprocess
import sys
from pathlib import Path

from quantum_lattice_models import create_model_spec
from quantum_lattice_models.registry import get_model_info, list_models, model_table

repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = repository_root / "results/notebooks"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
category_counts = {}
for name in list_models():
    category = get_model_info(name).category
    category_counts[category] = category_counts.get(category, 0) + 1

print("Registered model summary")
print("category      | models")
print("------------- | ------")
for category, count in sorted(category_counts.items()):
    print(f"{category:<13s} | {count:>6d}")
print(f"{'total':<13s} | {sum(category_counts.values()):>6d}")

info = get_model_info("ssh_model")
print("\nSSH model metadata")
print(f"  basis:       {info.basis}")
print(f"  dimension:   {info.dimension}")
print(f"  return type: {info.return_type}")
print("  defaults:")
for parameter, value in info.defaults.items():
    print(f"    {parameter:<8s} = {value}")

Registered model summary
category      | models
------------- | ------
hubbard       |      4
spin          |     18
tight_binding |     11
topological   |      5
user          |      2
total         |     40

SSH model metadata
  basis:       single particle
  dimension:   2*n_cells
  return type: LatticeHamiltonian
  defaults:
    n_cells  = 8
    t1       = 0.5
    t2       = 1.0


In [3]:
rows = model_table()
for category in sorted({row["category"] for row in rows}):
    print(f"{category.replace('_', ' ').title()} models")
    print("model                                      | dimension                 | return type")
    print(
        "------------------------------------------ | "
        "------------------------- | -----------------------"
    )
    for row in rows:
        if row["category"] == category:
            print(f"{row['name']:<42s} | {row['dimension']:<25s} | {row['return_type']}")
    print()

Hubbard models
model                                      | dimension                 | return type
------------------------------------------ | ------------------------- | -----------------------
bose_hubbard_chain                         | (max_occupancy+1)**n_sites | LatticeHamiltonian
bose_hubbard_chain_sparse                  | (max_occupancy+1)**n_sites | scipy.sparse.csr_matrix
fermi_hubbard_chain                        | 2**(2*n_sites)            | LatticeHamiltonian
fermi_hubbard_chain_sparse                 | 2**(2*n_sites)            | scipy.sparse.csr_matrix

Spin models
model                                      | dimension                 | return type
------------------------------------------ | ------------------------- | -----------------------
heisenberg_chain                           | 2**n_sites                | DenseHamiltonian
heisenberg_chain_sector_sparse             | comb(n_sites,(n_sites-magnetization)//2) | SpinSectorHamiltonian
heisenberg_chain_sparse     

In [4]:
models = subprocess.run(
    [sys.executable, "-m", "quantum_lattice_models.cli", "models", "--json"],
    check=True,
    capture_output=True,
    text=True,
)
print("CLI model discovery")
print("  JSON characters:", len(models.stdout))
print("  starts with:", models.stdout.splitlines()[0])

CLI model discovery
  JSON characters: 20671
  starts with: [


In [5]:
model_path = output_dir / "ssh_topological.json"
model = create_model_spec(
    "ssh_model",
    parameters={"n_cells": 4, "t1": 0.35, "t2": 1.0, "periodic": False},
)
model.save(model_path)

inspection = subprocess.run(
    [sys.executable, "-m", "quantum_lattice_models.cli", "inspect", str(model_path)],
    check=True,
    capture_output=True,
    text=True,
)
print("Portable SSH model")
print(f"  file: {model_path.relative_to(repository_root)}")
print(f"  local degrees: {len(model.local_degrees)}")
print(f"  interactions: {len(model.interactions)}")
has_dimension = '"dimension"' in inspection.stdout
print(f"  inspection contains dimension: {has_dimension}")

Portable SSH model
  file: results/notebooks/ssh_topological.json
  local degrees: 8
  interactions: 7
  inspection contains dimension: True


In [6]:
spectrum = subprocess.run(
    [
        sys.executable,
        "-m",
        "quantum_lattice_models.cli",
        "spectrum",
        str(model_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
energies = [float(line) for line in spectrum.stdout.splitlines() if line.strip()]
print("SSH spectrum from portable model file")
print("index | energy")
print("--- | ---")
for index, energy in enumerate(energies):
    print(f"{index:>5d} | {energy: .6f}")

SSH spectrum from portable model file
index | energy
--- | ---
    0 | -1.280566
    1 | -1.086119
    2 | -0.818731
    3 | -0.013178
    4 |  0.013178
    5 |  0.818731
    6 |  1.086119
    7 |  1.280566


In [7]:
plot_path = output_dir / "cli_ssh_spectrum.png"
bundle_path = output_dir / "ssh_topological.bundle"
subprocess.run(
    [
        sys.executable,
        "-m",
        "quantum_lattice_models.cli",
        "plot",
        "--model",
        "ssh_model",
        "--n-cells",
        "4",
        "--t1",
        "0.35",
        "--t2",
        "1.0",
        "--output",
        str(plot_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "quantum_lattice_models.cli",
        "export",
        str(model_path),
        "--artifact",
        "bundle",
        "--output",
        str(bundle_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print("CLI artifacts")
print(
    f"  plot: {plot_path.relative_to(repository_root)} ({plot_path.stat().st_size / 1024:.1f} KiB)"
)
print(f"  bundle files: {', '.join(sorted(path.name for path in bundle_path.iterdir()))}")

CLI artifacts
  plot: results/notebooks/cli_ssh_spectrum.png (24.2 KiB)
  bundle files: lattice.json, manifest.json, matrix.npz, metadata.json, model.json
